# HSRIS - Hybrid Semantic Retrieval and Intelligence System
### Assignment 3: Multi-Stage NLP Pipeline for Customer Support Ticket Retrieval

**Course:** Data Science for Software Engineering  
**Batch:** SE-6B  
**Group Members:**
- Ali Naqi (23F-3052)
- Muhammad Aamir (23F-3073)

**Platform:** Kaggle (GPU T4 x2) | **Framework:** Base PyTorch + NumPy | **Dataset:** Customer Support Tickets (~8,470 records)

---
**Pipeline Overview:**
1. **Part 1** - Categorical Encoders (Label + One-Hot) from scratch
2. **Part 2** - Sparse TF-IDF Representation (Custom Tokenizer, BoW, N-Grams)
3. **Part 3** - Dense GloVe Embeddings (TF-IDF Weighted Mean Pooling)
4. **Task 1** - Full Pipeline and Index Alignment
5. **Task 2** - Hybrid Search (alpha * TF-IDF + (1-alpha) * GloVe)
6. **Task 3** - Dual GPU Performance Optimization
7. **Visualization** - Side-by-side comparison, Execution Time Plot
8. **Evaluation** - Precision@5, 5 Qualitative Examples
9. **Streamlit App** - Interactive Demo

In [ ]:
# Cell 1 — Environment Check & GPU Verification
import torch
import os
import time
import gc

print("=" * 60)
print("HSRIS — Environment Check")
print("=" * 60)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}           : {torch.cuda.get_device_name(i)}")
        print(f"  Memory        : {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device    : {device}")
print("=" * 60)

## Cell 2 — Library Imports

In [ ]:
# Cell 2 — Import All Required Libraries
# NOTE: No sklearn imports allowed — everything is built from scratch
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
import gc
import time
from collections import Counter
import matplotlib.pyplot as plt

# Verify no sklearn contamination
print("All libraries imported successfully.")
print("sklearn is NOT imported — all encoders/vectorizers are custom-built.")

## Cell 3 - Load Dataset

Update the path below to match your Kaggle dataset input path.
Search for 'Customer Support Ticket Dataset' on Kaggle and add it as a data source.

In [ ]:
# Cell 3 - Load and Prepare Dataset
# UPDATE THIS PATH to match your Kaggle dataset location
DATASET_PATH = '/kaggle/input/datasets/waseemalastal/customer-support-ticket-dataset/customer_support_tickets.csv'

df = pd.read_csv(DATASET_PATH)
print(f"Raw dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Keep only the required fields
REQUIRED_COLS = ['Ticket Description', 'Ticket Subject', 'Ticket Priority', 'Ticket Type', 'Ticket Channel']
df = df[REQUIRED_COLS].dropna().reset_index(drop=True)

print(f"\nCleaned dataset shape: {df.shape}")
print(f"\nSample data:")
df.head(3)

## Part 1: Categorical Foundation (The Encoders)

### Cell 4 — Custom Label Encoder (Ticket Priority)
Map `Ticket Priority` (Low, Medium, High) to ordinal integers.
- Low → 0, Medium → 1, High → 2
- Unseen categories → -1 (safe fallback)

In [ ]:
# Cell 4 — Custom Label Encoder (from scratch — NO sklearn)
class CustomLabelEncoder:
    """
    Ordinal Label Encoder built from pure Python.
    Maps categorical labels to integer values.
    Handles unseen categories with a default value of -1.
    """
    def __init__(self):
        self.mapping = {}
        self.inverse_mapping = {}

    def fit(self, labels):
        """Learn the mapping from unique sorted labels to integers."""
        unique_labels = sorted(set(labels))
        self.mapping = {label: idx for idx, label in enumerate(unique_labels)}
        self.inverse_mapping = {idx: label for label, idx in self.mapping.items()}
        return self

    def transform(self, labels):
        """Transform labels to integers. Unseen labels get -1."""
        return np.array([self.mapping.get(label, -1) for label in labels])

    def inverse_transform(self, indices):
        """Convert integer indices back to original labels."""
        return [self.inverse_mapping.get(idx, 'UNKNOWN') for idx in indices]

    def __repr__(self):
        return f"CustomLabelEncoder(mapping={self.mapping})"


# Fit and transform Ticket Priority
priority_encoder = CustomLabelEncoder()
priority_encoder.fit(['Low', 'Medium', 'High'])  # Explicit ordinal order
priority_labels = priority_encoder.transform(df['Ticket Priority'])
priority_tensor = torch.tensor(priority_labels, dtype=torch.long).to(device)

print(f"Priority Mapping : {priority_encoder.mapping}")
print(f"Priority Tensor  : shape={priority_tensor.shape}, device={priority_tensor.device}")
print(f"Sample values    : {priority_labels[:10]}")
print(f"\n--- OOV Test ---")
print(f"Encoding ['Low', 'Critical', 'High'] → {priority_encoder.transform(['Low', 'Critical', 'High'])}")
print(f"'Critical' maps to -1 (unseen category handled safely)")

### Cell 5 — Custom One-Hot Encoder (Ticket Channel)
Convert `Ticket Channel` (e.g., Email, Phone, Chat, Social Media) into binary vector representation.
- Unseen categories → all-zeros vector (safe fallback)

In [ ]:
# Cell 5 — Custom One-Hot Encoder (from scratch — NO sklearn)
class CustomOneHotEncoder:
    """
    One-Hot Encoder built from pure Python + NumPy.
    Converts categorical values into binary vectors.
    Unseen categories during inference get an all-zeros vector.
    """
    def __init__(self):
        self.categories = []
        self.cat2idx = {}

    def fit(self, data):
        """Learn all unique categories from the training data."""
        self.categories = sorted(set(data))
        self.cat2idx = {cat: idx for idx, cat in enumerate(self.categories)}
        return self

    def transform(self, data):
        """Transform data to one-hot encoded binary matrix."""
        n_samples = len(data)
        n_categories = len(self.categories)
        matrix = np.zeros((n_samples, n_categories), dtype=np.float32)
        for i, val in enumerate(data):
            if val in self.cat2idx:
                matrix[i, self.cat2idx[val]] = 1.0
            # else: all-zeros row (unseen category fallback)
        return matrix

    def get_category_name(self, index):
        """Get category name from index."""
        if 0 <= index < len(self.categories):
            return self.categories[index]
        return 'UNKNOWN'

    def __repr__(self):
        return f"CustomOneHotEncoder(categories={self.categories})"


# Fit and transform Ticket Channel
channel_encoder = CustomOneHotEncoder()
channel_encoder.fit(df['Ticket Channel'])
channel_matrix = channel_encoder.transform(df['Ticket Channel'])
channel_tensor = torch.tensor(channel_matrix, dtype=torch.float32).to(device)

print(f"Channel Categories : {channel_encoder.categories}")
print(f"Channel Tensor     : shape={channel_tensor.shape}, device={channel_tensor.device}")
print(f"\nSample one-hot vectors:")
for i in range(min(3, len(df))):
    print(f"  '{df.iloc[i]['Ticket Channel']}' → {channel_matrix[i]}")

print(f"\n--- OOV Test ---")
test_oov = channel_encoder.transform(['Email', 'Telegram', 'Phone'])
print(f"Encoding ['Email', 'Telegram', 'Phone']:")
print(f"  'Telegram' row is all-zeros: {test_oov[1]}  (unseen category handled)")

## Part 2: Sparse Representation (Keyword Retrieval)

### Cell 6 — Custom Regex Tokenizer & N-Gram Generator
- Regex-based tokenizer: lowercasing + punctuation removal
- Sliding window for bigrams and trigrams (e.g., "not_working", "is_not_working")

In [ ]:
# Cell 6 — Custom Regex Tokenizer + N-Gram Generator (from scratch)
def tokenize(text):
    """
    Regex-based tokenizer:
    1. Convert to lowercase
    2. Extract only alphabetic tokens (removes punctuation, numbers)
    """
    text = str(text).lower()
    tokens = re.findall(r'[a-z]+', text)
    return tokens


def get_ngrams(tokens, n):
    """Generate n-grams using a sliding window approach."""
    return ['_'.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def tokenize_with_ngrams(text, ngram_range=(1, 3)):
    """
    Full tokenization pipeline:
    - Unigrams (single words)
    - Bigrams (2-word sequences) — captures context like 'not_working'
    - Trigrams (3-word sequences) — captures context like 'cannot_login_account'
    """
    tokens = tokenize(text)
    all_tokens = list(tokens)  # Start with unigrams
    for n in range(ngram_range[0] + 1, ngram_range[1] + 1):
        all_tokens.extend(get_ngrams(tokens, n))
    return all_tokens


# Demo
sample_text = "I cannot login to my account. Password reset is NOT working!"
tokens = tokenize(sample_text)
bigrams = get_ngrams(tokens, 2)
trigrams = get_ngrams(tokens, 3)
all_tokens = tokenize_with_ngrams(sample_text)

print(f"Original  : {sample_text}")
print(f"Unigrams  : {tokens}")
print(f"Bigrams   : {bigrams}")
print(f"Trigrams  : {trigrams}")
print(f"All tokens: {len(all_tokens)} total")

### Cell 7 — Build Vocabulary (Top 5,000 Tokens)
Count all tokens across all ticket descriptions and keep the top 5,000 most frequent.

In [ ]:
# Cell 7 — Build Vocabulary (BoW: Top 5000 Tokens)
TOP_K = 5000

all_token_counts = Counter()
tokenized_docs = []

for desc in df['Ticket Description']:
    toks = tokenize_with_ngrams(str(desc))
    tokenized_docs.append(toks)
    all_token_counts.update(toks)

print(f"Total unique tokens (incl. n-grams): {len(all_token_counts)}")

# Keep top 5000
vocab = [tok for tok, _ in all_token_counts.most_common(TOP_K)]
vocab2idx = {tok: i for i, tok in enumerate(vocab)}

print(f"Vocabulary size (top-K): {len(vocab)}")
print(f"\nTop 20 tokens:")
for tok, count in all_token_counts.most_common(20):
    print(f"  '{tok}': {count}")

### Cell 8 — Term-Frequency Matrix (Sparse)

> ⚠️ **CRITICAL:** Using `torch.sparse_coo_tensor` to avoid RAM crash. A dense 8470×5000 float matrix would consume too much Kaggle RAM.

In [ ]:
# Cell 8 — Build Sparse Term-Frequency Matrix
def build_sparse_tf(tokenized_docs, vocab2idx, vocab_size):
    """
    Build a sparse TF (Term Frequency) matrix using torch.sparse_coo_tensor.
    TF = count(term in doc) / total(terms in doc)
    """
    rows, cols, vals = [], [], []

    for doc_idx, tokens in enumerate(tokenized_docs):
        counts = Counter(tokens)
        total = sum(counts.values())
        if total == 0:
            continue
        for tok, cnt in counts.items():
            if tok in vocab2idx:
                rows.append(doc_idx)
                cols.append(vocab2idx[tok])
                vals.append(cnt / total)  # Term Frequency

    indices = torch.tensor([rows, cols], dtype=torch.long)
    values = torch.tensor(vals, dtype=torch.float32)
    shape = (len(tokenized_docs), vocab_size)
    return torch.sparse_coo_tensor(indices, values, shape)


tf_sparse = build_sparse_tf(tokenized_docs, vocab2idx, TOP_K)
print(f"Sparse TF Matrix:")
print(f"  Shape        : {tf_sparse.shape}")
print(f"  Non-zeros    : {tf_sparse._nnz()}")
print(f"  Density      : {tf_sparse._nnz() / (tf_sparse.shape[0] * tf_sparse.shape[1]) * 100:.2f}%")
print(f"  Storage type : torch.sparse_coo_tensor ✓")

### Cell 9 — IDF Computation & TF-IDF Transformation
- IDF = log((N + 1) / (df + 1)) + 1 (smoothed)
- TF-IDF = TF × IDF, L2-normalized per row
- Stored as sparse tensor after computation

In [ ]:
# Cell 9 — Compute IDF and Build TF-IDF Matrix
N = len(tokenized_docs)

# Document frequency: count how many docs contain each token
df_counts = torch.zeros(TOP_K)
for toks in tokenized_docs:
    unique_toks = set(toks)
    for tok in unique_toks:
        if tok in vocab2idx:
            df_counts[vocab2idx[tok]] += 1

# Smoothed IDF formula: log((N+1)/(df+1)) + 1
idf = torch.log((N + 1) / (df_counts + 1)) + 1
print(f"IDF vector: shape={idf.shape}, min={idf.min():.3f}, max={idf.max():.3f}")

# Apply IDF to sparse TF matrix
# We need to densify temporarily for element-wise IDF multiplication
tf_dense_temp = tf_sparse.to_dense()
tfidf_dense = tf_dense_temp * idf.unsqueeze(0)  # broadcast IDF across rows

# L2 normalize each row (document vector)
norms = tfidf_dense.norm(dim=1, keepdim=True).clamp(min=1e-9)
tfidf_dense = tfidf_dense / norms

# Convert back to sparse for memory-efficient storage
tfidf_sparse = tfidf_dense.to_sparse()

# Free temporary dense matrix
del tf_dense_temp
gc.collect()

print(f"\nTF-IDF Matrix:")
print(f"  Shape      : {tfidf_sparse.shape}")
print(f"  Non-zeros  : {tfidf_sparse._nnz()}")
print(f"  Normalized : L2-norm per row ✓")
print(f"  Storage    : torch.sparse_coo_tensor ✓")

## Part 3: Dense Semantic Layer (Neural Embeddings)

### Cell 10 — Load GloVe 300d into nn.Embedding Layer

> **Instructions:** Add GloVe 300d as a Kaggle dataset. Search for 'glove6b300d' or similar.
> Update `GLOVE_PATH` below to match your data source path.

- Index 0 = `<UNK>` token (zero vector for OOV handling)
- Weights are frozen (no gradient computation)

In [ ]:
# Cell 10 — Load GloVe Vectors into nn.Embedding
# ⚠️ UPDATE THIS PATH to match your Kaggle GloVe data source
# For 300d: search 'glove-global-vectors-for-word-representation' on Kaggle
# For 50d (if 300d unavailable): use the path below
import glob

# Auto-detect GloVe file
glove_candidates = glob.glob('/kaggle/input/**/glove*.txt', recursive=True)
# Also check known dataset path
import os as _os
_known = '/kaggle/input/datasets/thanakomsn/glove6b300dtxt/glove.6B.300d.txt'
if _os.path.exists(_known) and _known not in glove_candidates:
    glove_candidates.insert(0, _known)
print("Found GloVe files:", glove_candidates)

# Pick the largest dimensional file available (prefer 300d > 200d > 100d > 50d)
GLOVE_PATH = None
for dim_pref in ['300d', '200d', '100d', '50d']:
    for f_path in glove_candidates:
        if dim_pref in f_path:
            GLOVE_PATH = f_path
            break
    if GLOVE_PATH:
        break

if GLOVE_PATH is None and glove_candidates:
    GLOVE_PATH = glove_candidates[0]

print(f"Using GloVe file: {GLOVE_PATH}")

# Auto-detect embedding dimension from first line
with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
    first_line = f.readline().split()
    EMB_DIM = len(first_line) - 1  # first element is word, rest are vector values
print(f"Detected embedding dimension: {EMB_DIM}")

print(f"Loading GloVe {EMB_DIM}d vectors... (this may take 1-2 minutes)")
start_time = time.time()

glove_vocab = {}
with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        vec = np.array(parts[1:], dtype=np.float32)
        if vec.shape[0] == EMB_DIM:  # safety check
            glove_vocab[word] = vec

load_time = time.time() - start_time
print(f"GloVe loaded: {len(glove_vocab):,} words in {load_time:.1f}s")

# Build embedding matrix
# Index 0 = <UNK> (zero vector — OOV handling strategy)
# Indices 1..N = first 50,000 GloVe words
MAX_GLOVE_WORDS = 50000
glove_tokens = ['<UNK>'] + list(glove_vocab.keys())[:MAX_GLOVE_WORDS]
glove_tok2idx = {w: i for i, w in enumerate(glove_tokens)}

emb_matrix = np.zeros((len(glove_tokens), EMB_DIM), dtype=np.float32)
for i, w in enumerate(glove_tokens[1:], 1):
    emb_matrix[i] = glove_vocab[w]

# Create frozen nn.Embedding layer
embedding_layer = nn.Embedding(len(glove_tokens), EMB_DIM, padding_idx=0)
embedding_layer.weight.data.copy_(torch.tensor(emb_matrix))
embedding_layer.weight.requires_grad = False  # Freeze pretrained weights
embedding_layer = embedding_layer.to(device)

# Free raw GloVe dict to save RAM
del glove_vocab
gc.collect()

print(f"\nEmbedding Layer:")
print(f"  Vocab size     : {len(glove_tokens):,} (including <UNK>)")
print(f"  Embedding dim  : {EMB_DIM}")
print(f"  Device         : {next(embedding_layer.parameters()).device}")
print(f"  Frozen         : {not embedding_layer.weight.requires_grad} ✓")
print(f"  OOV strategy   : <UNK> → zero vector at index 0 ✓")

# Helper function
def get_glove_indices(tokens):
    """Map tokens to GloVe indices. Unknown tokens → 0 (<UNK>)."""
    return [glove_tok2idx.get(t, 0) for t in tokens]

### Cell 11 — TF-IDF Weighted Mean Pooling (Sentence Vectors)

> ⚠️ **CRITICAL:** Simple mean pooling is **forbidden**. We use TF-IDF weights so rare technical keywords dominate over common filler words.

This prevents **semantic dilution** — words like "kernel" or "billing" carry more weight than "the" or "hello".

In [ ]:
# Cell 11 — TF-IDF Weighted GloVe Sentence Vectors (Anti-Semantic-Dilution)
def get_sentence_vector(tokens, tfidf_row_dict, embedding_layer, device):
    """
    Compute a TF-IDF weighted average of GloVe word embeddings.
    This gives higher weight to rare/important words and lower weight to common words.
    
    Args:
        tokens: list of word tokens
        tfidf_row_dict: {token: tfidf_weight} for the document
        embedding_layer: nn.Embedding with GloVe weights
        device: torch device
    Returns:
        Fixed 300-dim sentence vector
    """
    total_weight = 0.0
    weighted_sum = torch.zeros(EMB_DIM, device=device)
    indices = get_glove_indices(tokens)

    for tok, idx in zip(tokens, indices):
        if idx == 0:  # Skip <UNK> tokens (zero vector anyway)
            continue
        weight = tfidf_row_dict.get(tok, 1e-5)  # tiny weight for tokens not in TF-IDF vocab
        emb = embedding_layer(torch.tensor([idx], device=device))[0]
        weighted_sum += weight * emb
        total_weight += weight

    if total_weight == 0:
        return weighted_sum  # zero vector if no valid tokens
    return weighted_sum / total_weight


# Build sentence vectors for all documents
print("Building TF-IDF weighted GloVe sentence vectors...")
print("This may take 10-20 minutes for ~8,470 documents.")
start_time = time.time()

sentence_vectors = []
tfidf_dense_cpu = tfidf_dense.cpu()

for i, toks in enumerate(tokenized_docs):
    # Build TF-IDF weight dict for this document (only unigrams for GloVe)
    row = tfidf_dense_cpu[i]
    nonzero_indices = row.nonzero(as_tuple=True)[0]
    row_dict = {vocab[j]: row[j].item() for j in nonzero_indices}

    # Only use unigram tokens for GloVe (n-grams won't be in GloVe vocab)
    unigram_toks = tokenize(str(df.iloc[i]['Ticket Description']))
    vec = get_sentence_vector(unigram_toks, row_dict, embedding_layer, device)
    sentence_vectors.append(vec.unsqueeze(0))

    if (i + 1) % 1000 == 0:
        elapsed = time.time() - start_time
        print(f"  Processed {i+1}/{len(tokenized_docs)} docs ({elapsed:.1f}s)")

# Concatenate all sentence vectors into a matrix [N, 300]
glove_matrix = torch.cat(sentence_vectors, dim=0)
# L2 normalize for cosine similarity
glove_matrix = F.normalize(glove_matrix, dim=1)

total_time = time.time() - start_time
print(f"\nDone! GloVe matrix shape: {glove_matrix.shape}")
print(f"Total time: {total_time:.1f}s")
print(f"L2 normalized: ✓")

# Cleanup
del sentence_vectors, tfidf_dense_cpu
gc.collect()
torch.cuda.empty_cache()

## Task 1: The Pipeline — Index Alignment

### Cell 12 — Verify All Representations Are Aligned
Ensure every representation has the same row count and aligns with the original DataFrame index.

In [ ]:
# Cell 12 — Pipeline Alignment Verification
print("=" * 60)
print("PIPELINE ALIGNMENT CHECK")
print("=" * 60)

n_records = len(df)
assert n_records == tfidf_dense.shape[0], f"TF-IDF rows mismatch: {tfidf_dense.shape[0]} vs {n_records}"
assert n_records == glove_matrix.shape[0], f"GloVe rows mismatch: {glove_matrix.shape[0]} vs {n_records}"
assert n_records == len(priority_tensor), f"Priority tensor mismatch: {len(priority_tensor)} vs {n_records}"
assert n_records == channel_tensor.shape[0], f"Channel tensor mismatch: {channel_tensor.shape[0]} vs {n_records}"

print(f"✅ All representations aligned: {n_records} records")
print(f"")
print(f"  DataFrame       : {df.shape}")
print(f"  Label Encoding  : {priority_tensor.shape}  (Ticket Priority)")
print(f"  One-Hot Encoding: {channel_tensor.shape}  (Ticket Channel)")
print(f"  TF-IDF (sparse) : {tfidf_sparse.shape}    (Ticket Description)")
print(f"  TF-IDF (dense)  : {tfidf_dense.shape}     (for similarity computation)")
print(f"  GloVe Embeddings: {glove_matrix.shape}   (Ticket Description)")
print(f"")
print("All three encoding types (One-Hot, TF-IDF, Dense) are generated ✓")
print("Metadata (Priority/Channel) aligns with text vectors ✓")

## Task 2: Hybrid Search Logic

### Cell 13 — Hybrid Retrieval Function

**Formula:** `FinalScore = α × (TF-IDF Score) + (1 − α) × (GloVe Score)` where α = 0.4

In [ ]:
# Cell 13 — Hybrid Search Function
ALPHA = 0.4  # 0.4 TF-IDF + 0.6 GloVe (semantic-leaning)

def hybrid_search(query, alpha=ALPHA, top_k=5):
    """
    Hybrid retrieval: combines keyword-based TF-IDF with semantic GloVe similarity.
    
    FinalScore = α × (TF-IDF cosine sim) + (1 - α) × (GloVe cosine sim)
    
    Args:
        query: search query string
        alpha: weight for TF-IDF (0.0 = pure GloVe, 1.0 = pure TF-IDF)
        top_k: number of results to return
    Returns:
        DataFrame with top-k matching tickets and their scores
    """
    # --- Encode query with TF-IDF ---
    q_toks = tokenize_with_ngrams(query)
    q_tf = Counter(q_toks)
    q_total = sum(q_tf.values()) or 1

    q_tfidf = torch.zeros(TOP_K, device=device)
    for tok, cnt in q_tf.items():
        if tok in vocab2idx:
            q_tfidf[vocab2idx[tok]] = (cnt / q_total) * idf[vocab2idx[tok]].to(device)
    q_tfidf = F.normalize(q_tfidf.unsqueeze(0), dim=1)

    # --- Encode query with GloVe (TF-IDF weighted) ---
    q_unigrams = tokenize(query)
    q_row_dict = {}
    for j in q_tfidf[0].nonzero(as_tuple=True)[0]:
        if j.item() < len(vocab):
            q_row_dict[vocab[j.item()]] = q_tfidf[0, j].item()
    q_glove = get_sentence_vector(q_unigrams, q_row_dict, embedding_layer, device)
    q_glove = F.normalize(q_glove.unsqueeze(0), dim=1)

    # --- Cosine Similarities ---
    tfidf_on_device = tfidf_dense.to(device)
    glove_on_device = glove_matrix.to(device)

    tfidf_scores = torch.mm(q_tfidf, tfidf_on_device.T).squeeze(0)
    glove_scores = torch.mm(q_glove, glove_on_device.T).squeeze(0)

    # --- Hybrid Score ---
    final_scores = alpha * tfidf_scores + (1 - alpha) * glove_scores
    top_indices = final_scores.topk(top_k).indices.cpu().tolist()

    # Build results
    results = df.iloc[top_indices][['Ticket Description', 'Ticket Subject', 
                                      'Ticket Type', 'Ticket Priority', 'Ticket Channel']].copy()
    results['Hybrid_Score'] = [final_scores[i].item() for i in top_indices]
    results['TFIDF_Score'] = [tfidf_scores[i].item() for i in top_indices]
    results['GloVe_Score'] = [glove_scores[i].item() for i in top_indices]

    return results


# Test the hybrid search
print("=" * 60)
print("HYBRID SEARCH TEST")
print("=" * 60)
test_query = "I cannot login to my account, password reset not working"
results = hybrid_search(test_query)
print(f"Query: '{test_query}'\n")
for i, (_, row) in enumerate(results.iterrows(), 1):
    print(f"{i}. [{row['Ticket Type']}] (Score: {row['Hybrid_Score']:.4f})")
    print(f"   TF-IDF: {row['TFIDF_Score']:.4f} | GloVe: {row['GloVe_Score']:.4f}")
    print(f"   {str(row['Ticket Description'])[:100]}...")
    print()

## Task 3: Performance Optimization (Dual GPU)

### Cell 14 — DataParallel Similarity Model
Wrap the similarity computation in `nn.DataParallel` to split batch search across both T4 GPUs.

In [ ]:
# Cell 14 — GPU-Parallel Similarity Model with DataParallel
class SimilarityModel(nn.Module):
    """
    Wraps hybrid similarity computation for DataParallel across dual T4 GPUs.
    Stores corpus TF-IDF and GloVe matrices as non-trainable parameters.
    """
    def __init__(self, corpus_tfidf, corpus_glove, alpha=0.4):
        super().__init__()
        self.corpus_tfidf = nn.Parameter(corpus_tfidf, requires_grad=False)
        self.corpus_glove = nn.Parameter(corpus_glove, requires_grad=False)
        self.alpha = alpha

    def forward(self, q_tfidf_batch, q_glove_batch):
        """
        Compute hybrid similarity for a batch of queries.
        Args:
            q_tfidf_batch: [B, vocab_size] — TF-IDF query vectors
            q_glove_batch: [B, 300] — GloVe query vectors
        Returns:
            [B, N] — hybrid similarity scores for each query against all docs
        """
        tfidf_sim = torch.mm(q_tfidf_batch, self.corpus_tfidf.T)
        glove_sim = torch.mm(q_glove_batch, self.corpus_glove.T)
        return self.alpha * tfidf_sim + (1 - self.alpha) * glove_sim


# Initialize model
sim_model = SimilarityModel(tfidf_dense.to(device), glove_matrix.to(device))

# Wrap with DataParallel if multiple GPUs available
if torch.cuda.device_count() > 1:
    print(f"Using DataParallel across {torch.cuda.device_count()} GPUs ✓")
    sim_model = nn.DataParallel(sim_model)
else:
    print("Single GPU mode")

sim_model = sim_model.to(device)
print(f"SimilarityModel ready on {device}")

### Cell 15 — Batch Query Encoding & GPU Timing
Encode 100 test queries and measure execution time across different batch sizes.

In [ ]:
# Cell 15 — Encode 100 Test Queries & Run Timing Loop

# Sample 100 test queries from the dataset
np.random.seed(42)
test_query_indices = np.random.choice(len(df), 100, replace=False)
test_queries = df.iloc[test_query_indices]['Ticket Description'].tolist()

def encode_query_batch(queries):
    """Encode a list of queries into TF-IDF and GloVe tensors."""
    tfidf_batch, glove_batch = [], []
    for q in queries:
        q_toks = tokenize_with_ngrams(str(q))
        # TF-IDF encoding
        qt = Counter(q_toks)
        total = sum(qt.values()) or 1
        qv = torch.zeros(TOP_K)
        for tok, cnt in qt.items():
            if tok in vocab2idx:
                qv[vocab2idx[tok]] = (cnt / total) * idf[vocab2idx[tok]]
        qv = F.normalize(qv.unsqueeze(0), dim=1)

        # GloVe encoding (TF-IDF weighted)
        q_unigrams = tokenize(str(q))
        rd = {}
        for j in qv[0].nonzero(as_tuple=True)[0]:
            if j.item() < len(vocab):
                rd[vocab[j.item()]] = qv[0, j].item()
        gv = get_sentence_vector(q_unigrams, rd, embedding_layer, device)
        gv = F.normalize(gv.unsqueeze(0), dim=1)

        tfidf_batch.append(qv)
        glove_batch.append(gv.cpu())

    return torch.cat(tfidf_batch).to(device), torch.cat(glove_batch).to(device)

print("Encoding 100 test queries...")
start = time.time()
q_tfidf_all, q_glove_all = encode_query_batch(test_queries)
print(f"Encoding done in {time.time()-start:.1f}s")
print(f"  TF-IDF batch: {q_tfidf_all.shape}")
print(f"  GloVe batch : {q_glove_all.shape}")

# Timing loop over different batch sizes
batch_sizes = [10, 25, 50, 75, 100]
exec_times = []

print(f"\nGPU Timing Results:")
print("-" * 40)
for bs in batch_sizes:
    # Warm-up
    with torch.no_grad():
        _ = sim_model(q_tfidf_all[:bs], q_glove_all[:bs])
    torch.cuda.synchronize()

    # Timed run
    start = time.time()
    with torch.no_grad():
        scores = sim_model(q_tfidf_all[:bs], q_glove_all[:bs])
    torch.cuda.synchronize()
    elapsed = time.time() - start
    exec_times.append(elapsed)
    print(f"  Batch {bs:>3}: {elapsed:.6f}s  |  {bs/elapsed:.0f} queries/sec")

### Cell 16 — Execution Time vs Batch Size Plot (Deliverable)
Training log: GPU execution time across different query batch sizes.

In [ ]:
# Cell 16 — Execution Time vs Query Batch Size Plot
plt.figure(figsize=(10, 6))

# Main line
plt.plot(batch_sizes, exec_times, 'o-', color='#2E75B6', linewidth=2.5,
         markersize=10, markerfacecolor='#1B4F72', markeredgecolor='white',
         markeredgewidth=2, label='Dual T4 GPU')

# Annotations
for bs, t in zip(batch_sizes, exec_times):
    plt.annotate(f'{t*1000:.2f}ms', (bs, t),
                 textcoords="offset points", xytext=(0, 15),
                 fontsize=9, ha='center', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.xlabel('Query Batch Size', fontsize=13, fontweight='bold')
plt.ylabel('Execution Time (seconds)', fontsize=13, fontweight='bold')
plt.title('GPU Execution Time vs Query Batch Size (Dual T4 x2)', fontsize=15, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(fontsize=12)
plt.xticks(batch_sizes)
plt.tight_layout()
plt.savefig('execution_time_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as 'execution_time_plot.png' ✓")

## Visualization Module

### Cell 17 — Side-by-Side: TF-IDF vs GloVe Comparison
Display results from keyword-based (TF-IDF) and semantic-based (GloVe) retrieval for a given query.

In [ ]:
# Cell 17 — Side-by-Side TF-IDF vs GloVe Comparison Utility
def compare_methods(query, top_k=5):
    """Display side-by-side comparison of TF-IDF only vs GloVe only results."""
    # TF-IDF only (alpha=1.0)
    tfidf_results = hybrid_search(query, alpha=1.0, top_k=top_k)
    # GloVe only (alpha=0.0)
    glove_results = hybrid_search(query, alpha=0.0, top_k=top_k)

    print(f"{'='*80}")
    print(f"QUERY: '{query}'")
    print(f"{'='*80}")

    print(f"\n📊 TF-IDF Results (keyword overlap):")
    print(f"{'-'*40}")
    for i, (_, row) in enumerate(tfidf_results.iterrows(), 1):
        print(f"  {i}. [{row['Ticket Type']}] (score: {row['TFIDF_Score']:.4f})")
        print(f"     {str(row['Ticket Description'])[:90]}...")

    print(f"\n🧠 GloVe Results (semantic meaning):")
    print(f"{'-'*40}")
    for i, (_, row) in enumerate(glove_results.iterrows(), 1):
        print(f"  {i}. [{row['Ticket Type']}] (score: {row['GloVe_Score']:.4f})")
        print(f"     {str(row['Ticket Description'])[:90]}...")
    print()


# Demo comparison
compare_methods("I need help with money transfer and payment issues")

## Quantitative Evaluation

### Cell 18 — Precision@5
For each test query, check how many of the top-5 retrieved tickets share the same `Ticket Type` as the query.

In [ ]:
# Cell 18 — Precision@5 Evaluation
def precision_at_k(query_idx, k=5, alpha=ALPHA):
    """
    Compute Precision@K for a single query.
    Precision@K = (# of top-K results with same Ticket Type as query) / K
    """
    query_text = str(df.iloc[query_idx]['Ticket Description'])
    true_type = df.iloc[query_idx]['Ticket Type']

    results = hybrid_search(query_text, alpha=alpha, top_k=k + 1)
    # Exclude self-match (query itself)
    results = results[results.index != query_idx].head(k)
    matches = (results['Ticket Type'] == true_type).sum()
    return matches / k


# Evaluate on 200 random queries
np.random.seed(123)
eval_indices = np.random.choice(len(df), 200, replace=False)

print("Computing Precision@5 on 200 random queries...")
start_time = time.time()
p5_scores = []
for i, idx in enumerate(eval_indices):
    p5 = precision_at_k(idx)
    p5_scores.append(p5)
    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/200 queries...")

mean_p5 = np.mean(p5_scores)
eval_time = time.time() - start_time

print(f"\n{'='*50}")
print(f"PRECISION@5 RESULTS")
print(f"{'='*50}")
print(f"  Queries evaluated : 200")
print(f"  Mean Precision@5  : {mean_p5:.4f}")
print(f"  Std Precision@5   : {np.std(p5_scores):.4f}")
print(f"  Min / Max         : {np.min(p5_scores):.2f} / {np.max(p5_scores):.2f}")
print(f"  Evaluation time   : {eval_time:.1f}s")
print(f"  Alpha used        : {ALPHA}")
print(f"{'='*50}")

# Distribution plot
plt.figure(figsize=(8, 4))
plt.hist(p5_scores, bins=6, color='#2E75B6', edgecolor='white', alpha=0.8)
plt.xlabel('Precision@5', fontsize=12)
plt.ylabel('Number of Queries', fontsize=12)
plt.title(f'Precision@5 Distribution (Mean = {mean_p5:.4f})', fontsize=14, fontweight='bold')
plt.axvline(mean_p5, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_p5:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('precision_at_5_distribution.png', dpi=150)
plt.show()

### Cell 19 — 5 Qualitative Examples: GloVe Outperforms TF-IDF
These examples demonstrate cases where semantic understanding (GloVe) finds better matches than keyword overlap (TF-IDF).

In [ ]:
# Cell 19 — 5 Qualitative Examples Where GloVe Beats TF-IDF
qualitative_queries = [
    ("I have a problem with money transfer",
     "GloVe matches 'billing/payment' tickets; TF-IDF misses if exact word absent"),
    ("my device keeps freezing",
     "GloVe matches 'system crash/hang' tickets; TF-IDF misses synonyms"),
    ("I cannot get into my account",
     "GloVe matches 'login/authentication' tickets; TF-IDF misses paraphrase"),
    ("internet is very slow",
     "GloVe matches 'network performance' tickets; TF-IDF needs exact 'slow'"),
    ("I am unhappy with my recent purchase",
     "GloVe matches 'refund/dissatisfied' tickets; TF-IDF misses sentiment words"),
]

print("=" * 80)
print("5 QUALITATIVE EXAMPLES: SEMANTIC SEARCH (GloVe) vs KEYWORD SEARCH (TF-IDF)")
print("=" * 80)

for i, (query, explanation) in enumerate(qualitative_queries, 1):
    print(f"\n{'─'*80}")
    print(f"EXAMPLE {i}: '{query}'")
    print(f"Expected advantage: {explanation}")
    print(f"{'─'*80}")
    compare_methods(query, top_k=3)

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE — 5 qualitative examples where GloVe outperforms TF-IDF")
print("=" * 80)

## Streamlit App — Interactive Demo

### Cell 20 — Generate & Launch Streamlit App
Writes a `streamlit_app.py` file and launches it with a public tunnel.

**Features:**
- Text area for entering a new ticket description
- Slider to adjust α value (0.0 = GloVe only → 1.0 = TF-IDF only)
- Displays predicted Ticket Type and top 3 similar past resolutions

In [ ]:
# Cell 20 -- Streamlit App (Write file + Launch)
# Install streamlit for the interactive demo
import subprocess
subprocess.run(['pip', 'install', 'streamlit', '-q'])

# Save all necessary data for the Streamlit app
import pickle

app_data = {
    'df': df,
    'vocab': vocab,
    'vocab2idx': vocab2idx,
    'idf': idf.cpu(),
    'tfidf_dense': tfidf_dense.cpu(),
    'glove_matrix': glove_matrix.cpu(),
    'emb_matrix': emb_matrix,
    'glove_tokens': glove_tokens,
    'glove_tok2idx': glove_tok2idx,
    'TOP_K': TOP_K,
    'EMB_DIM': EMB_DIM,
    'priority_encoder_mapping': priority_encoder.mapping,
    'channel_encoder_categories': channel_encoder.categories,
}

with open('hsris_app_data.pkl', 'wb') as f:
    pickle.dump(app_data, f)
print("App data saved to hsris_app_data.pkl")

# Write the Streamlit app file with advanced CSS styling
streamlit_code = '''
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
import pickle
import os
from collections import Counter

st.set_page_config(page_title="HSRIS | Hybrid Search", page_icon="H", layout="wide")

# Advanced CSS Styling
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&family=JetBrains+Mono:wght@400;500;600&display=swap');
    html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
    .stApp { background: linear-gradient(135deg, #0a0a1a 0%, #0d1b2a 30%, #1b263b 60%, #0d1b2a 100%); }
    #MainMenu, footer, header { visibility: hidden; }

    .hero-banner {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
        background-size: 200% 200%; animation: gf 6s ease infinite;
        border-radius: 24px; padding: 48px 40px; margin-bottom: 32px;
        position: relative; overflow: hidden;
        box-shadow: 0 20px 60px rgba(102,126,234,0.3);
    }
    .hero-banner::before {
        content:''; position:absolute; top:0; left:0; right:0; bottom:0;
        background: url("data:image/svg+xml,%3Csvg width='60' height='60' viewBox='0 0 60 60' xmlns='http://www.w3.org/2000/svg'%3E%3Cg fill='none'%3E%3Cg fill='%23ffffff' fill-opacity='0.05'%3E%3Cpath d='M36 34v-4h-2v4h-4v2h4v4h2v-4h4v-2h-4zm0-30V0h-2v4h-4v2h4v4h2V6h4V4h-4zM6 34v-4H4v4H0v2h4v4h2v-4h4v-2H6zM6 4V0H4v4H0v2h4v4h2V6h4V4H6z'/%3E%3C/g%3E%3C/g%3E%3C/svg%3E");
        opacity: 0.4;
    }
    @keyframes gf { 0%{background-position:0% 50%} 50%{background-position:100% 50%} 100%{background-position:0% 50%} }
    .hero-banner h1 { font-size:2.4rem; font-weight:900; color:#fff; margin:0 0 8px; position:relative; text-shadow:0 2px 20px rgba(0,0,0,0.2); }
    .hero-banner .sub { color:rgba(255,255,255,0.85); font-size:1.05rem; margin:0; position:relative; }
    .hero-banner .badges { display:flex; gap:10px; margin-top:20px; position:relative; flex-wrap:wrap; }
    .hb { background:rgba(255,255,255,0.15); backdrop-filter:blur(12px); border:1px solid rgba(255,255,255,0.2); border-radius:100px; padding:6px 16px; font-size:0.8rem; font-weight:600; color:#fff; }

    .glass { background:rgba(255,255,255,0.04); backdrop-filter:blur(20px); border:1px solid rgba(255,255,255,0.08); border-radius:20px; padding:28px 32px; margin-bottom:20px; transition:all .3s; }
    .glass:hover { border-color:rgba(102,126,234,0.3); box-shadow:0 8px 32px rgba(102,126,234,0.15); transform:translateY(-2px); }
    .glass h3 { font-size:1.1rem; font-weight:700; color:#e0e6ff; margin:0 0 4px; }
    .glass .cs { font-size:0.82rem; color:rgba(255,255,255,0.45); margin-bottom:16px; }

    .rc { background:linear-gradient(135deg,rgba(255,255,255,0.05),rgba(255,255,255,0.02)); border:1px solid rgba(255,255,255,0.08); border-radius:16px; padding:24px 28px; margin-bottom:16px; transition:all .3s; position:relative; overflow:hidden; }
    .rc::before { content:''; position:absolute; top:0; left:0; width:4px; height:100%; border-radius:4px 0 0 4px; }
    .rc.r1::before { background:linear-gradient(to bottom,#ffd700,#ffaa00); }
    .rc.r2::before { background:linear-gradient(to bottom,#c0c0c0,#a0a0a0); }
    .rc.r3::before { background:linear-gradient(to bottom,#cd7f32,#a0622e); }
    .rc:hover { border-color:rgba(102,126,234,0.25); transform:translateY(-1px); }
    .rr { font-size:0.75rem; font-weight:700; text-transform:uppercase; letter-spacing:0.08em; margin-bottom:8px; }
    .r1 .rr { color:#ffd700; } .r2 .rr { color:#c0c0c0; } .r3 .rr { color:#cd7f32; }
    .rs { font-family:'JetBrains Mono',monospace; font-size:1.6rem; font-weight:700; background:linear-gradient(135deg,#667eea,#764ba2); -webkit-background-clip:text; -webkit-text-fill-color:transparent; margin-bottom:12px; }
    .rt { display:inline-block; background:linear-gradient(135deg,#667eea,#764ba2); color:white; padding:4px 14px; border-radius:100px; font-size:0.78rem; font-weight:600; margin-bottom:12px; }
    .mg { display:grid; grid-template-columns:repeat(3,1fr); gap:8px; margin-bottom:14px; }
    .mi { background:rgba(255,255,255,0.04); border-radius:10px; padding:10px 14px; text-align:center; }
    .ml { font-size:0.68rem; text-transform:uppercase; letter-spacing:0.06em; color:rgba(255,255,255,0.4); margin-bottom:2px; }
    .mv { font-size:0.88rem; font-weight:600; color:#e0e6ff; }
    .rd { font-size:0.88rem; color:rgba(255,255,255,0.65); line-height:1.6; border-top:1px solid rgba(255,255,255,0.06); padding-top:14px; }

    .pb { background:linear-gradient(135deg,rgba(102,126,234,0.15),rgba(118,75,162,0.15)); border:1px solid rgba(102,126,234,0.25); border-radius:16px; padding:24px 28px; margin-bottom:24px; text-align:center; animation:fsi .5s ease-out; }
    .pl { font-size:0.78rem; text-transform:uppercase; letter-spacing:0.1em; color:rgba(255,255,255,0.5); margin-bottom:6px; }
    .pv { font-size:1.8rem; font-weight:800; background:linear-gradient(135deg,#667eea,#764ba2,#f093fb); -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
    @keyframes fsi { from{opacity:0;transform:translateY(12px)} to{opacity:1;transform:translateY(0)} }

    .ab { display:flex; justify-content:space-between; align-items:center; background:rgba(255,255,255,0.04); border-radius:12px; padding:14px 20px; margin-top:16px; }
    .al { font-size:0.82rem; color:rgba(255,255,255,0.5); }
    .av { font-family:'JetBrains Mono',monospace; font-weight:700; font-size:1.1rem; color:#e0e6ff; }

    .sr { display:flex; gap:12px; margin-bottom:28px; flex-wrap:wrap; }
    .sp { background:rgba(255,255,255,0.04); border:1px solid rgba(255,255,255,0.08); border-radius:100px; padding:8px 20px; display:flex; align-items:center; gap:8px; }
    .sn { font-family:'JetBrains Mono',monospace; font-weight:700; font-size:0.95rem; color:#e0e6ff; }
    .sl { font-size:0.78rem; color:rgba(255,255,255,0.4); }

    .fb { background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.08); border-radius:12px; padding:16px 20px; font-family:'JetBrains Mono',monospace; font-size:0.9rem; color:#c5ceff; text-align:center; margin-bottom:20px; }
    .af { margin-top:60px; padding:30px 0; border-top:1px solid rgba(255,255,255,0.06); text-align:center; }
    .af p { color:rgba(255,255,255,0.3); font-size:0.82rem; margin:4px 0; }
    .af .fbr { font-weight:700; background:linear-gradient(135deg,#667eea,#764ba2); -webkit-background-clip:text; -webkit-text-fill-color:transparent; }

    .stTextArea textarea { background:rgba(255,255,255,0.04)!important; border:1px solid rgba(255,255,255,0.1)!important; border-radius:14px!important; color:#e0e6ff!important; font-family:'Inter',sans-serif!important; font-size:0.95rem!important; padding:16px!important; }
    .stTextArea textarea:focus { border-color:rgba(102,126,234,0.5)!important; box-shadow:0 0 0 3px rgba(102,126,234,0.15)!important; }
    div.stButton > button[kind="primary"] { background:linear-gradient(135deg,#667eea,#764ba2)!important; color:white!important; border:none!important; border-radius:14px!important; padding:14px 28px!important; font-weight:700!important; font-size:1rem!important; box-shadow:0 4px 20px rgba(102,126,234,0.3)!important; transition:all .3s!important; }
    div.stButton > button[kind="primary"]:hover { transform:translateY(-2px)!important; box-shadow:0 8px 30px rgba(102,126,234,0.45)!important; }
    .sh { font-size:1.1rem; font-weight:700; color:#e0e6ff; margin-bottom:16px; padding-bottom:10px; border-bottom:1px solid rgba(255,255,255,0.06); }
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_data():
    with open('hsris_app_data.pkl', 'rb') as f:
        data = pickle.load(f)
    emb = nn.Embedding(len(data['glove_tokens']), data['EMB_DIM'], padding_idx=0)
    emb.weight.data.copy_(torch.tensor(data['emb_matrix']))
    emb.weight.requires_grad = False
    data['embedding_layer'] = emb
    return data

data = load_data()
df = data['df']; vocab = data['vocab']; vocab2idx = data['vocab2idx']
idf = data['idf']; tfidf_dense = data['tfidf_dense']; glove_matrix = data['glove_matrix']
embedding_layer = data['embedding_layer']; glove_tok2idx = data['glove_tok2idx']
TOP_K = data['TOP_K']; EMB_DIM = data['EMB_DIM']; device = torch.device('cpu')

def tokenize(text): return re.findall(r'[a-z]+', str(text).lower())
def get_ngrams(tokens, n): return ['_'.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
def tokenize_with_ngrams(text):
    tokens = tokenize(text); a = list(tokens); a.extend(get_ngrams(tokens,2)); a.extend(get_ngrams(tokens,3)); return a
def get_glove_indices(tokens): return [glove_tok2idx.get(t,0) for t in tokens]
def get_sentence_vector(tokens, rd):
    tw=0.0; ws=torch.zeros(EMB_DIM)
    for tok,idx in zip(tokens, get_glove_indices(tokens)):
        if idx==0: continue
        w=rd.get(tok,1e-5); ws+=w*embedding_layer(torch.tensor([idx]))[0]; tw+=w
    return ws/tw if tw>0 else ws

def hybrid_search(query, alpha=0.4, top_k=5):
    qt=Counter(tokenize_with_ngrams(query)); total=sum(qt.values()) or 1
    qv=torch.zeros(TOP_K)
    for tok,cnt in qt.items():
        if tok in vocab2idx: qv[vocab2idx[tok]]=(cnt/total)*idf[vocab2idx[tok]]
    qv=F.normalize(qv.unsqueeze(0),dim=1)
    q_uni=tokenize(query)
    rd={vocab[j]:qv[0,j].item() for j in qv[0].nonzero(as_tuple=True)[0] if j.item()<len(vocab)}
    gv=F.normalize(get_sentence_vector(q_uni,rd).unsqueeze(0),dim=1)
    ts=torch.mm(qv,tfidf_dense.T).squeeze(0); gs=torch.mm(gv,glove_matrix.T).squeeze(0)
    fs=alpha*ts+(1-alpha)*gs; top_i=fs.topk(top_k).indices.tolist()
    res=df.iloc[top_i][['Ticket Description','Ticket Subject','Ticket Type','Ticket Priority','Ticket Channel']].copy()
    res['Score']=[fs[i].item() for i in top_i]
    return res, df.iloc[top_i]['Ticket Type'].mode()[0]

# Hero Banner
st.markdown("""<div class="hero-banner"><h1>HSRIS</h1><p class="sub">Hybrid Semantic Retrieval and Intelligence System</p>
<div class="badges"><span class="hb">PyTorch</span><span class="hb">GloVe 300D</span><span class="hb">TF-IDF</span><span class="hb">Dual GPU</span>
<span class="hb">Ali Naqi (23F-3052)</span><span class="hb">Muhammad Aamir (23F-3073)</span></div></div>""", unsafe_allow_html=True)
n=len(df)
st.markdown(f"""<div class="sr"><div class="sp"><span class="sn">{n:,}</span><span class="sl">tickets</span></div>
<div class="sp"><span class="sn">{TOP_K:,}</span><span class="sl">vocab</span></div>
<div class="sp"><span class="sn">{EMB_DIM}D</span><span class="sl">GloVe</span></div>
<div class="sp"><span class="sn">SE-6B</span><span class="sl">batch</span></div></div>""", unsafe_allow_html=True)
st.markdown('<div class="fb">FinalScore = &alpha; &times; CosineSim(TF-IDF) + (1 - &alpha;) &times; CosineSim(GloVe)</div>', unsafe_allow_html=True)

c1,c2=st.columns([2.5,1])
with c1:
    st.markdown('<div class="glass"><h3>Enter Ticket Description</h3><div class="cs">Type a customer support ticket to find similar past resolutions</div></div>', unsafe_allow_html=True)
    query=st.text_area("Q",height=130,placeholder="e.g. I cannot login to my account...",label_visibility="collapsed")
with c2:
    st.markdown('<div class="glass"><h3>Search Parameters</h3><div class="cs">Adjust the hybrid weighting</div></div>', unsafe_allow_html=True)
    alpha=st.slider("A",0.0,1.0,0.4,0.05,label_visibility="collapsed")
    st.markdown(f'<div class="ab"><div><div class="al">TF-IDF</div><div class="av">{alpha:.0%}</div></div><div><div class="al">GloVe</div><div class="av">{1-alpha:.0%}</div></div></div>', unsafe_allow_html=True)

if st.button("Search", type="primary", use_container_width=True) and query.strip():
    results, pt = hybrid_search(query, alpha=alpha, top_k=3)
    st.markdown(f'<div class="pb"><div class="pl">Predicted Ticket Type</div><div class="pv">{pt}</div></div>', unsafe_allow_html=True)
    st.markdown('<div class="sh">Top 3 Similar Past Resolutions</div>', unsafe_allow_html=True)
    rl=["1st Match","2nd Match","3rd Match"]
    for i,(_,row) in enumerate(results.iterrows()):
        rc=f"r{i+1}"
        st.markdown(f"""<div class="rc {rc}"><div class="rr">{rl[i]}</div><div class="rs">{row['Score']:.4f}</div>
        <span class="rt">{row['Ticket Type']}</span>
        <div class="mg"><div class="mi"><div class="ml">Priority</div><div class="mv">{row['Ticket Priority']}</div></div>
        <div class="mi"><div class="ml">Channel</div><div class="mv">{row['Ticket Channel']}</div></div>
        <div class="mi"><div class="ml">Subject</div><div class="mv">{str(row['Ticket Subject'])[:30]}</div></div></div>
        <div class="rd">{row['Ticket Description']}</div></div>""", unsafe_allow_html=True)

st.markdown("""<div class="af"><p><span class="fbr">HSRIS</span> -- Hybrid Semantic Retrieval and Intelligence System</p>
<p>Assignment 3 | Data Science for Software Engineering | SE-6B</p>
<p>Ali Naqi (23F-3052) | Muhammad Aamir (23F-3073)</p></div>""", unsafe_allow_html=True)
'''

with open('streamlit_app.py', 'w') as f:
    f.write(streamlit_code)
print("Streamlit app written to streamlit_app.py")

# Launch instructions
print("\n" + "="*60)
print("TO LAUNCH THE STREAMLIT APP:")
print("="*60)
print("Run in a new cell:")
print("  !streamlit run streamlit_app.py &>/content/logs.txt &")
print("  !npx localtunnel --port 8501")
print("\nThis will give you a public URL to share.")
print("Alternatively, deploy to Streamlit Cloud for a permanent link.")